# 02. 해설집 지식베이스 구축

목표: Google Sheets의 `완제품 모델 해설집`을 원문과 구조를 보존한 통합 지식베이스로 변환한다.

- 입력: 공유 드라이브의 Google Sheets 문서
- 처리: 시트별 컬럼 보존, 공통 메타데이터 부여, 검색용 텍스트 생성
- 출력: 통합 CSV, JSONL, 시트별 정리본을 공유 드라이브의 `완제품 모델` 폴더에 저장

이 단계에서는 모델 학습이나 HS Code 정답 생성은 하지 않는다. 원문에 없는 분류 기준을 추론해 추가하지 않는다.

In [ ]:
!pip -q install pandas openpyxl google-api-python-client google-auth-httplib2

In [ ]:
from google.colab import auth
import google.auth
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
from pathlib import Path
from io import BytesIO
import pandas as pd
import json, re, os

auth.authenticate_user()
credentials, project = google.auth.default()
credentials.refresh(Request())
drive_service = build('drive', 'v3', credentials=credentials)
FOLDER_ID = '1GAOGfn5zoLCpgQu62at8WC24gVbCmrma'
folder = drive_service.files().get(fileId=FOLDER_ID, supportsAllDrives=True, fields='id,name,capabilities(canAddChildren)').execute()
print(folder)

In [ ]:
SPREADSHEET_ID = '1HN6a_AJBzejNvhlrld_TEM7DpPzkWjZTYjdq1UEhwno'
file_info = drive_service.files().get(fileId=SPREADSHEET_ID, supportsAllDrives=True, fields='id,name,mimeType').execute()
print(file_info)

if file_info['mimeType'] == 'application/vnd.google-apps.spreadsheet':
    request = drive_service.files().export_media(fileId=SPREADSHEET_ID, mimeType='application/vnd.openxmlformats-officedocument.spreadsheetml.sheet')
elif file_info['mimeType'] == 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet':
    request = drive_service.files().get_media(fileId=SPREADSHEET_ID, supportsAllDrives=True)
else:
    raise ValueError(f"지원하지 않는 파일 형식: {file_info['mimeType']}")
excel_buffer = BytesIO()
downloader = MediaIoBaseDownload(excel_buffer, request)
done = False
while not done:
    status, done = downloader.next_chunk()
excel_buffer.seek(0)
xls = pd.ExcelFile(excel_buffer)
print('시트 목록:', xls.sheet_names)

In [ ]:
sheet_frames = {sheet: pd.read_excel(BytesIO(excel_buffer.getvalue()), sheet_name=sheet, dtype=str).fillna('') for sheet in xls.sheet_names}
for sheet, frame in sheet_frames.items():
    print(sheet, frame.shape)
    display(frame.head(3))

In [ ]:
sheet_frames = {sheet: pd.read_excel(excel_buffer if False else BytesIO(excel_buffer.getvalue()), sheet_name=sheet, dtype=str).fillna('') for sheet in xls.sheet_names}
for sheet, frame in sheet_frames.items():
    print(sheet, frame.shape)
    display(frame.head(3))

In [ ]:
records = []
for sheet_name, frame in sheet_frames.items():
    for row_number, row in frame.iterrows():
        item = {str(k): str(v).strip() for k, v in row.to_dict().items()}
        item = {k: v for k, v in item.items() if v and v.lower() != 'nan'}
        searchable = [f'{k}: {v}' for k, v in item.items() if k not in {'원문행'}]
        records.append({
            'knowledge_id': item.get('규칙_ID', f'{sheet_name}-{row_number + 1}'),
            'source_sheet': sheet_name,
            'source_row': str(row_number + 2),
            'hs_code': item.get('HS코드') or item.get('관련_HS코드') or item.get('대상_HS코드') or item.get('검토_HS코드') or item.get('호코드', ''),
            'rule_type': item.get('규칙유형') or item.get('구분') or item.get('규칙_성격') or item.get('해설서_구분', ''),
            'source_page_or_line': item.get('근거_구간') or item.get('근거') or item.get('원문행', ''),
            'document_text': '\n'.join(searchable),
            'raw_fields_json': json.dumps(item, ensure_ascii=False)
        })
knowledge_df = pd.DataFrame(records)
print('총 지식 레코드:', len(knowledge_df))
print(knowledge_df.groupby('source_sheet').size())
display(knowledge_df.head())

In [ ]:
for col in ['knowledge_id', 'source_sheet', 'document_text']:
    if knowledge_df[col].duplicated().any():
        print('중복 확인:', col, int(knowledge_df[col].duplicated().sum()))
knowledge_df['document_text_length'] = knowledge_df['document_text'].str.len()
print(knowledge_df['document_text_length'].describe())

In [ ]:
output_dir = Path('/content/hs_knowledge_base')
output_dir.mkdir(parents=True, exist_ok=True)
knowledge_df.to_csv(output_dir / 'hs_knowledge_base.csv', index=False, encoding='utf-8-sig')
knowledge_df.to_json(output_dir / 'hs_knowledge_base.jsonl', orient='records', lines=True, force_ascii=False)
for sheet_name, frame in sheet_frames.items():
    safe_name = re.sub(r'[^가-힣A-Za-z0-9_-]+', '_', sheet_name)
    frame.to_csv(output_dir / f'{safe_name}.csv', index=False, encoding='utf-8-sig')
print(sorted(p.name for p in output_dir.iterdir()))

In [ ]:
uploaded = []
for path in output_dir.iterdir():
    metadata = {'name': path.name, 'parents': [FOLDER_ID]}
    media = MediaFileUpload(str(path), mimetype='application/json' if path.suffix == '.jsonl' else 'text/csv', resumable=True)
    result = drive_service.files().create(body=metadata, media_body=media, supportsAllDrives=True, fields='id,name').execute()
    uploaded.append(result)
for item in uploaded:
    print('업로드 완료:', item['name'], item['id'])

## 다음 단계

이 노트북의 산출물을 검토한 뒤 `03_text_analysis.ipynb`에서 다음을 진행한다.

1. 규칙 유형별 검색 대상 구성
2. TF-IDF 키워드 검색
3. Sentence Embedding 기반 의미 검색
4. 검색 결과에 원문 출처와 페이지/행 정보 연결

주의: `소호체계`는 코드 계층 정보이며, 나머지 시트는 규정·설명·판단 기준이다. 이후 검색 및 모델 입력에서도 이 차이를 유지한다.